In [1]:
# https://ec.europa.eu/eurostat/databrowser/view/nasa_10_f_bs__custom_21212140/default/table

In [2]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt

In [ ]:
# Use pycountry to get the iso3 codes
import pycountry

# PyCountry mapping
records = []
for c in pycountry.countries:
    records.append({
        "country_name": c.name,
        "iso3":         c.alpha_3,
        "iso2":         c.alpha_2,
        "iso_numeric":  c.numeric,
    })

df_iso = pd.DataFrame(records)
df_iso.shape

(249, 4)

In [25]:
EU27 = [
    "AUT","BEL","BGR","CZE","DNK","EST","FIN","FRA","DEU","GRC",
    "HUN","ITA","LVA","LTU","NLD","POL","PRT","ROU","SVK",
    "SVN","ESP","SWE","HRV","CYP","MLT","IRL","LUX"
]
len(EU27)

27

In [18]:
filepath = "../Raw/nasa_10_f_bs__custom_21212140_spreadsheet.xlsx"

# Get sheets
xls = pd.ExcelFile(filepath)
print(xls.sheet_names)

['Summary', 'Sheet 1', 'Sheet 2', 'Sheet 3', 'Sheet 4', 'Sheet 5', 'Sheet 6', 'Sheet 7', 'Sheet 8', 'Sheet 9', 'Sheet 10']


/Users/jesper/Desktop/CBS/Thesis 1/Jesper-Liedholm-Thesis-Code/.venv/lib/python3.12/site-packages/openpyxl/styles/stylesheet.py:237: UserWarning: Workbook contains no default style, apply openpyxl's default
  warn("Workbook contains no default style, apply openpyxl's default")


We want:
- Sheet 3-5. Only Assets are relevant

In [19]:
df_sh3 = pd.read_excel(xls, 'Sheet 3', skiprows=11)
df_sh4 = pd.read_excel(xls, 'Sheet 4', skiprows=11)
df_sh5 = pd.read_excel(xls, 'Sheet 5', skiprows=11)

# Drop all columns with Unnamed in the name
df_sh3 = df_sh3.loc[:, ~df_sh3.columns.str.contains('Unnamed')]
df_sh4 = df_sh4.loc[:, ~df_sh4.columns.str.contains('Unnamed')]
df_sh5 = df_sh5.loc[:, ~df_sh5.columns.str.contains('Unnamed')]

# Drop the first row, the one that says 'GEO (Labels)'
df_sh3 = df_sh3.drop(0)
df_sh4 = df_sh4.drop(0)
df_sh5 = df_sh5.drop(0)

# Rename TIME to Country
df_sh3 = df_sh3.rename(columns={'TIME': 'Country'})
df_sh4 = df_sh4.rename(columns={'TIME': 'Country'})
df_sh5 = df_sh5.rename(columns={'TIME': 'Country'})

# Make country column string and strip whitespace
df_sh3["Country"] = df_sh3["Country"].astype(str).str.strip()
df_sh4["Country"] = df_sh4["Country"].astype(str).str.strip()
df_sh5["Country"] = df_sh5["Country"].astype(str).str.strip()

# Remove the Country that includes 'European Union'
df_sh3 = df_sh3[~df_sh3['Country'].str.contains('European Union')]
df_sh4 = df_sh4[~df_sh4['Country'].str.contains('European Union')]
df_sh5 = df_sh5[~df_sh5['Country'].str.contains('European Union')]

# Get yr columns and convert to numeric, coercing errors to NaN
yr_cols_sh3 = [col for col in df_sh3.columns if len(col) == 4 and col.strip().startswith('2')]
yr_cols_sh4 = [col for col in df_sh4.columns if len(col) == 4 and col.strip().startswith('2')]
yr_cols_sh5 = [col for col in df_sh5.columns if len(col) == 4 and col.strip().startswith('2')]

for col in yr_cols_sh3:
    df_sh3[col] = pd.to_numeric(df_sh3[col], errors='coerce')
for col in yr_cols_sh4:
    df_sh4[col] = pd.to_numeric(df_sh4[col], errors='coerce')
for col in yr_cols_sh5:
    df_sh5[col] = pd.to_numeric(df_sh5[col], errors='coerce')

display(df_sh3.head(3))
display(df_sh4.head(3))
display(df_sh5.head(3))

,Country,2016,2017,2018,2019,2020,2021,2022,2023,2024,2025
2,Belgium,652205.4,655170.5,654922.8,656174.3,712366.1,720245.6,644239.4,692318.3,705827.7,728251.5
3,Bulgaria,27638.3,25754.5,27296.8,31801.9,29735.2,30211.2,26910.5,44952.3,47627.2,NaN
4,Czechia,126022.0,134320.7,137512.3,143735.2,161458.6,180089.4,185696.8,207230.1,220116.6,NaN


,Country,2016,2017,2018,2019,2020,2021,2022,2023,2024,2025
2,Belgium,229796.8,244407.9,203393.6,247853.3,254875.3,320082.3,263877.5,281127.6,310677.1,361265.4
3,Bulgaria,4999.1,5290.9,5277.3,5196.7,5560.9,7254.0,7478.4,8179.5,9423.9,NaN
4,Czechia,18291.4,27383.0,29322.5,35131.6,41056.4,57376.1,49733.6,58920.9,67254.6,NaN


,Country,2016,2017,2018,2019,2020,2021,2022,2023,2024,2025
2,Belgium,328150.5,373523.7,346120.1,414956.7,438648.5,510665.1,459027.1,512256.9,577711.7,641023.7
3,Bulgaria,1504.7,1857.1,2034.6,2189.3,2999.0,4486.3,4240.7,5161.5,6303.9,NaN
4,Czechia,22850.9,27876.6,28536.4,33213.8,35675.3,46943.2,52204.0,66403.0,82388.5,NaN


In [20]:
# Now we wnat to get the total portfolio holdings. So we
# add the three different sheets.
# Debt Securities + Listed Shares + Investment fund shares/units

# Melt each into long format
df_sh3_melt = df_sh3.melt(id_vars='Country', value_vars=yr_cols_sh3, var_name='Year', value_name='Debt_Securities')
df_sh4_melt = df_sh4.melt(id_vars='Country', value_vars=yr_cols_sh4, var_name='Year', value_name='Listed_Shares')
df_sh5_melt = df_sh5.melt(id_vars='Country', value_vars=yr_cols_sh5, var_name='Year', value_name='Investment_Fund_Shares')

display(df_sh3_melt.head(3))
display(df_sh4_melt.head(3))
display(df_sh5_melt.head(3))

,Country,Year,Debt_Securities
0,Belgium,2016,652205.4
1,Bulgaria,2016,27638.3
2,Czechia,2016,126022.0


,Country,Year,Listed_Shares
0,Belgium,2016,229796.8
1,Bulgaria,2016,4999.1
2,Czechia,2016,18291.4


,Country,Year,Investment_Fund_Shares
0,Belgium,2016,328150.5
1,Bulgaria,2016,1504.7
2,Czechia,2016,22850.9


In [21]:
# Join on Country and Year
df_euro = df_sh3_melt.merge(df_sh4_melt, on=['Country', 'Year'], how='outer')
df_euro = df_euro.merge(df_sh5_melt, on=['Country', 'Year'], how='outer')

df_euro["Total Portfolio Holdings"] = df_euro["Debt_Securities"] + df_euro["Listed_Shares"] + df_euro["Investment_Fund_Shares"]

df_euro = df_euro.sort_values(by=["Total Portfolio Holdings"], ascending=False)

display(df_euro.head(3))
df_euro.shape

,Country,Year,Debt_Securities,Listed_Shares,Investment_Fund_Shares,Total Portfolio Holdings
128,Germany,2024,3943126.0,2017094.0,4170284.0,10130504.0
125,Germany,2021,4049884.0,2045892.0,3867948.0,9963724.0
127,Germany,2023,3845734.0,1860901.0,3781849.0,9488484.0


(420, 6)

In [22]:
# --- Domestic share ---
# Now we need to see what the domestic shares would be for each country
df_cpis = pd.read_csv('../Clean/IMF_CPIS.csv')
df_cpis.head()

,REF_AREA,REF_AREA_LABEL,INDICATOR,INDICATOR_LABEL,COMP_BREAKDOWN_1,COMP_BREAKDOWN_1_LABEL,COMP_BREAKDOWN_2,COMP_BREAKDOWN_2_LABEL,COMP_BREAKDOWN_3,COMP_BREAKDOWN_3_LABEL,UNIT_MEASURE,UNIT_MEASURE_LABEL,year,value,iso3_i,iso3_j
0,ARG,Argentina,IMF_CPIS_I_A_D_L_T_BP6,"Assets, Debt Securities",IMF_CNT_COUNTRY_1,World,IMF_CNT_SEC_T,Total Holdings,IMF_SEC_T,Sector: All sectors,USD,US dollars,1997,1.805003e+10,ARG,NaN
1,ARG,Argentina,IMF_CPIS_I_A_D_L_T_BP6,"Assets, Debt Securities",IMF_CNT_COUNTRY_111,United States,IMF_CNT_SEC_T,Total Holdings,IMF_SEC_T,Sector: All sectors,USD,US dollars,1997,3.558190e+09,ARG,USA
2,ARG,Argentina,IMF_CPIS_I_A_D_L_T_BP6,"Assets, Debt Securities",IMF_CNT_COUNTRY_112,United Kingdom,IMF_CNT_SEC_T,Total Holdings,IMF_SEC_T,Sector: All sectors,USD,US dollars,1997,1.285639e+06,ARG,GBR
3,ARG,Argentina,IMF_CPIS_I_A_D_L_T_BP6,"Assets, Debt Securities",IMF_CNT_COUNTRY_113,Guernsey,IMF_CNT_SEC_T,Total Holdings,IMF_SEC_T,Sector: All sectors,USD,US dollars,1997,0.000000e+00,ARG,GGY
4,ARG,Argentina,IMF_CPIS_I_A_D_L_T_BP6,"Assets, Debt Securities",IMF_CNT_COUNTRY_117,Jersey,IMF_CNT_SEC_T,Total Holdings,IMF_SEC_T,Sector: All sectors,USD,US dollars,1997,0.000000e+00,ARG,JEY


In [23]:
# Only keep total investments
# df_cpis["INDICATOR_LABEL"].value_counts()
df_cpis = df_cpis[df_cpis["INDICATOR_LABEL"] == "Assets, Total Investment"]

In [26]:
# Only countries in europe
df_cpis = df_cpis[df_cpis["REF_AREA"].isin(EU27)]

In [27]:
# CPIS is in USD, convert to millions
df_cpis['value'] = df_cpis['value'] / 1e6

In [28]:
# Group CPIS per iso3_i and year, summing the value
df_cpis_grouped = df_cpis.groupby(['iso3_i', 'year'])['value'].sum().reset_index()
df_cpis_grouped

,iso3_i,year,value
0,AUT,1997,1.018444e+05
1,AUT,2001,2.249144e+05
2,AUT,2002,3.049703e+05
3,AUT,2003,4.140695e+05
4,AUT,2004,5.290751e+05
...,...,...,...
583,SWE,2019,1.608374e+06
584,SWE,2020,1.932562e+06
585,SWE,2021,1.974681e+06
586,SWE,2022,1.548510e+06


In [29]:
# View cpis for germany
df_cpis_grouped[df_cpis_grouped['iso3_i'] == 'DEU'].head(3)

,iso3_i,year,value
117,DEU,2001,1.583233e+06
118,DEU,2002,1.795664e+06
119,DEU,2003,2.410254e+06


In [30]:
# Make sure they are comparable by getting currency exchange rate

# ECB historical daily FX rates: EUR base, e.g. USD = USD per 1 EUR
url = "https://www.ecb.europa.eu/stats/eurofxref/eurofxref-hist.zip"

fx = pd.read_csv(url)

# Clean
fx["Date"] = pd.to_datetime(fx["Date"])
fx = fx.sort_values("Date")

# Keep last available ECB business-day rate in each year
year_end_fx = (
    fx.dropna(subset=["USD"])
      .assign(year=lambda x: x["Date"].dt.year)
      .groupby("year", as_index=False)
      .tail(1)
      [["year", "Date", "USD"]]
      .rename(columns={
          "Date": "fx_date",
          "USD": "usd_per_eur"
      })
)

# Last 30 available years
year_end_fx = year_end_fx.tail(30).reset_index(drop=True)

# Conversion rate: million USD -> million EUR
year_end_fx["eur_per_usd"] = 1 / year_end_fx["usd_per_eur"]

year_end_fx.head()

,year,fx_date,usd_per_eur,eur_per_usd
0,1999,1999-12-30,1.0046,0.995421
1,2000,2000-12-29,0.9305,1.074691
2,2001,2001-12-28,0.8813,1.134687
3,2002,2002-12-31,1.0487,0.953562
4,2003,2003-12-31,1.2630,0.791766


In [31]:
# Convert Eurostat to USD using the year-end FX rate
df_euro["Year"] = df_euro["Year"].astype(int)

if "eur_per_usd" in df_euro.columns:
    df_euro = df_euro.drop(columns=["eur_per_usd"])
if "year" in df_euro.columns:
    df_euro = df_euro.drop(columns=["year"])

df_euro = df_euro.merge(year_end_fx[["year", "eur_per_usd"]], left_on="Year", right_on="year", how="left")
df_euro["Total Portfolio Holdings (EUR)"] = df_euro["Total Portfolio Holdings"] * df_euro["eur_per_usd"]
df_euro.head(3)

,Country,Year,Debt_Securities,Listed_Shares,Investment_Fund_Shares,Total Portfolio Holdings,year,eur_per_usd,Total Portfolio Holdings (EUR)
0,Germany,2024,3943126.0,2017094.0,4170284.0,10130504.0,2024,0.962557,9.751183e+06
1,Germany,2021,4049884.0,2045892.0,3867948.0,9963724.0,2021,0.882924,8.797213e+06
2,Germany,2023,3845734.0,1860901.0,3781849.0,9488484.0,2023,0.904977,8.586863e+06


In [15]:
df_euro = df_euro.merge(df_iso[["country_name", "iso3"]], left_on="Country", right_on="country_name", how="left")
df_euro.head(3)

,Country,Year,Debt_Securities,Listed_Shares,Investment_Fund_Shares,Total Portfolio Holdings,year,eur_per_usd,Total Portfolio Holdings (EUR),country_name,iso3
0,Germany,2024,3943126.0,2017094.0,4170284.0,10130504.0,2024,0.962557,9.751183e+06,Germany,DEU
1,Germany,2021,4049884.0,2045892.0,3867948.0,9963724.0,2021,0.882924,8.797213e+06,Germany,DEU
2,Germany,2023,3845734.0,1860901.0,3781849.0,9488484.0,2023,0.904977,8.586863e+06,Germany,DEU


In [16]:
df1 = df_euro[["iso3", "Year", "Total Portfolio Holdings (EUR)"]]
df2 = df_cpis_grouped[["iso3_i", "year", "value"]].rename(columns={"value": "CPIS_Value", "iso3_i": "iso3", "year": "Year"})

df_full = df1.merge(df2, on=["iso3", "Year"], how="inner")
df_full.head(3)

,iso3,Year,Total Portfolio Holdings (EUR),CPIS_Value
0,DEU,2021,8.797213e+06,2.874998e+07
1,DEU,2023,8.586863e+06,2.724067e+07
2,FRA,2021,7.812314e+06,1.757883e+07
